**V1 experiment.** This notebook contains the V1 implementation and experiment configuration. See [methods and evaluation](../docs/METHODS.md) for the implemented protocol.


# V2 — Diffusion-map + t-SNE embeddings for Level 1 and Level 2

Restores the V1 embedding figures in the V2 setup. For Level 2 we plot
four variants so the effect of Fix 5 (best params + connectivity fix)
and Fix 6 (chroma→MFCC-std features) on the manifold is visible.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib as mpl
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.neighbors import NearestNeighbors

PROJECT_ROOT = Path.cwd().resolve()
while not ((PROJECT_ROOT / "utils" / "graph.py").is_file() and
           (PROJECT_ROOT / "v2" / "utils" / "splits.py").is_file()):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Start Jupyter inside the cloned repository.")
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.graph import build_knn_graph, diffusion_map_embedding
from v2.utils.graph import ensure_connected
from v2.utils.runs import make_run_dir

V2 = PROJECT_ROOT / "v2"
# Per-run output folders for the embedding figures (one new run per
# notebook execution, in each level).
L1_RUN = make_run_dir(V2 / "Level1", notebook_label="embeddings_v2.ipynb")
L2_RUN = make_run_dir(V2 / "Level2", notebook_label="embeddings_v2.ipynb")
L1_FIG = L1_RUN / "figures"
L2_FIG = L2_RUN / "figures"
print(f"Level 1 run -> {L1_RUN}")
print(f"Level 2 run -> {L2_RUN}")


def two_panel_embedding(diff_xy, tsne_xy, labels, label_names, title_left, title_right,
                        save_path, figsize=(15, 7), legend_ncol=2, point_size=10):
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    cmap = mpl.colormaps.get_cmap("tab20").resampled(max(len(label_names), 4))
    for ax, coords, title in [(axes[0], diff_xy, title_left),
                              (axes[1], tsne_xy, title_right)]:
        for i, name in enumerate(label_names):
            m = labels == i
            ax.scatter(coords[m, 0], coords[m, 1], s=point_size, alpha=0.7,
                       color=cmap(i), label=name)
        ax.set_title(title); ax.set_xlabel("dim 1"); ax.set_ylabel("dim 2")
        if len(label_names) <= 12:
            ax.legend(fontsize=7, markerscale=1.5, loc="best", ncol=legend_ncol)
        else:
            ax.legend(fontsize=6, markerscale=1.5, loc="best", ncol=legend_ncol)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.close(fig)
    print("saved", save_path)


def sigma_for(k, label, Xz):
    if label == "self_tuning":
        return "self_tuning"
    nn = NearestNeighbors(n_neighbors=k + 1).fit(Xz)
    d, _ = nn.kneighbors(Xz)
    med = float(np.median(d[:, 1:]))
    factor = float(label.split("xmed")[0])
    return factor * med


## Level 1 — diffusion map + t-SNE on V1 features (k=10, σ=median)


In [ ]:
d1 = np.load(PROJECT_ROOT / "Level1" / "Data" / "level1_features.npz", allow_pickle=True)
X1 = d1["X"].astype(np.float32); y1 = d1["labels"].astype(np.int64)
label_names_1 = list(d1["label_names"])
Xz1 = StandardScaler().fit_transform(X1).astype(np.float32)

W1 = ensure_connected(build_knn_graph(Xz1, k=10, sigma="median"), Xz1)
diff1 = diffusion_map_embedding(W1, n_components=3, t=5)
tsne1 = TSNE(n_components=2, perplexity=30, init="pca", random_state=0).fit_transform(Xz1)
two_panel_embedding(
    diff1[:, :2], tsne1, y1, label_names_1,
    title_left="Level 1 V2 — Diffusion map (t=5, k=10, σ=median)",
    title_right="Level 1 V2 — t-SNE",
    save_path=L1_FIG / "embeddings_v2.png",
    legend_ncol=1,
)


## Level 2 — V1 features at default (k=10, σ=median)


In [ ]:
d2v1 = np.load(PROJECT_ROOT / "Level2" / "Data" / "level2_features.npz", allow_pickle=True)
X2v1 = d2v1["X"].astype(np.float32); y2 = d2v1["labels"].astype(np.int64)
label_names_2 = list(d2v1["label_names"])
Xz2v1 = StandardScaler().fit_transform(X2v1).astype(np.float32)

W2v1_def = ensure_connected(build_knn_graph(Xz2v1, k=10, sigma="median"), Xz2v1)
diff_def = diffusion_map_embedding(W2v1_def, n_components=3, t=5)
tsne2 = TSNE(n_components=2, perplexity=30, init="pca", random_state=0).fit_transform(Xz2v1)
two_panel_embedding(
    diff_def[:, :2], tsne2, y2, label_names_2,
    title_left="Level 2 V2 — Diffusion map (V1 feats, default k=10, σ=median, t=5)",
    title_right="Level 2 V2 — t-SNE (V1 feats)",
    save_path=L2_FIG / "embeddings_v2_default.png",
    legend_ncol=2,
    point_size=8,
)


## Level 2 — V1 features at Fix 5 best (k=5, σ=self_tuning, +connectivity)


In [ ]:
sig_v1_best = sigma_for(5, "self_tuning", Xz2v1)
W2v1_best = ensure_connected(build_knn_graph(Xz2v1, k=5, sigma=sig_v1_best), Xz2v1)
diff_v1_best = diffusion_map_embedding(W2v1_best, n_components=3, t=5)
two_panel_embedding(
    diff_v1_best[:, :2], tsne2, y2, label_names_2,
    title_left="Level 2 V2 — Diffusion map (V1 feats, Fix 5 best k=5, σ=self_tune, t=5)",
    title_right="Level 2 V2 — t-SNE (V1 feats; same coords as default)",
    save_path=L2_FIG / "embeddings_v2_best.png",
    legend_ncol=2,
    point_size=8,
)


## Level 2 — V2 features at Fix 6 best (k=3, σ=self_tuning, +connectivity)


In [ ]:
# V2 features are written by 02_preprocessing_v2.ipynb to Level2/Data/.
# (preprocessing isn't run-folder-versioned because it produces a single
# canonical features .npz that downstream notebooks consume.)
d2v2 = np.load(V2 / "Level2" / "Data" / "level2_features_v2.npz", allow_pickle=True)
X2v2 = d2v2["X"].astype(np.float32)
Xz2v2 = StandardScaler().fit_transform(X2v2).astype(np.float32)

sig_v2_best = sigma_for(3, "self_tuning", Xz2v2)
W2v2_best = ensure_connected(build_knn_graph(Xz2v2, k=3, sigma=sig_v2_best), Xz2v2)
diff_v2_best = diffusion_map_embedding(W2v2_best, n_components=3, t=5)
tsne2_v2 = TSNE(n_components=2, perplexity=30, init="pca", random_state=0).fit_transform(Xz2v2)
two_panel_embedding(
    diff_v2_best[:, :2], tsne2_v2, y2, label_names_2,
    title_left="Level 2 V2 — Diffusion map (V2 feats, Fix 6 best k=3, σ=self_tune, t=5)",
    title_right="Level 2 V2 — t-SNE (V2 feats)",
    save_path=L2_FIG / "embeddings_v2_features.png",
    legend_ncol=2,
    point_size=8,
)


## Level 2 — diffusion times t = 1, 5, 20 on V2 features at Fix 6 best params


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
cmap = mpl.colormaps.get_cmap("tab20").resampled(max(len(label_names_2), 4))
for ax, t in zip(axes, [1, 5, 20]):
    coords = diffusion_map_embedding(W2v2_best, n_components=2, t=t)
    for i, name in enumerate(label_names_2):
        m = y2 == i
        ax.scatter(coords[m, 0], coords[m, 1], s=8, alpha=0.6,
                   color=cmap(i), label=name)
    ax.set_title(f"diffusion map t={t} (V2 feats, k=3, σ=self_tune)")
    ax.set_xlabel("dim 1"); ax.set_ylabel("dim 2")
fig.tight_layout()
fig.savefig(L2_FIG / "diffusion_times_v2.png", dpi=150)
plt.close(fig)
print("saved", L2_FIG / "diffusion_times_v2.png")
